In [3]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline
import pandas as pd

In [18]:
def sentiment(data):
    label_mapping = {'positive': 1, 'neutral': 0, 'negative': -1}
    data['score'] = data['news'].apply(lambda x: label_mapping[nlp(x)[0]['label']])
    
    total_weight = data.groupby(['Datetime', 'ticker'])['weight'].transform('sum')
    data['weighted_score'] = data['score'] * data['weight']
    
    data['sentiment'] = data.groupby(['Datetime', 'ticker'])['weighted_score'].transform('sum') / total_weight
    result = data[['Datetime', 'ticker', 'sentiment']].drop_duplicates()
    result = result.sort_values(by=['ticker', 'Datetime']).reset_index(drop=True)
    
    data.drop(['score', 'weighted_score','sentiment'], axis=1, inplace=True)
    
    return result

In [19]:
model = BertForSequenceClassification.from_pretrained("ahmedrachid/FinancialBERT-Sentiment-Analysis", num_labels=3)
tokenizer = BertTokenizer.from_pretrained("ahmedrachid/FinancialBERT-Sentiment-Analysis")
nlp = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)

In [26]:
news_dict = dict({
    'Datetime':['2024-01-01 13:30:00+00:00','2024-10-28 13:33:00+00:00','2024-10-28 13:30:00+00:00'],
    'ticker':['AAPL']*2+['GOOG'],
    'news':['Outperform Expectations','Underperform Expectations', 'M&A'],
    'weight':[1,1,1]
})
news_df = pd.DataFrame(news_dict)

In [27]:
# Unit Test
nlp('Outperform Expectations')

[{'label': 'neutral', 'score': 0.9993125200271606}]

In [28]:
sentiment(news_df)

,Datetime,ticker,sentiment
0,2024-01-01 13:30:00+00:00,AAPL,0.0
1,2024-10-28 13:33:00+00:00,AAPL,0.0
2,2024-10-28 13:30:00+00:00,GOOG,0.0
